Introduction
============

This is interactive notebook regarding "Introduction to path planning". (Author: Björn Hein)

Version | Authors
------------ | -------------
0.2 | Dennis McNab, Benjamin Dilly, Anton Kisel


License is based on Creative Commons: Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) (pls. check: http://creativecommons.org/licenses/by-nc/4.0/)

This notebook imports all discussed algorithms and does a comparison

Important links are:

* General Info: http://www.python.org
* Python tutorial http://www.python.org/doc/tut/
* NetworkX http://networkx.github.io/
* NumPy and SciPy Guide http://docs.scipy.org/
* Matplotlib gallery http://matplotlib.sourceforge.net/gallery.html


Remember that:

* you have to press ctrl-return or shift-return, to execute the code in the code sections, only then the variables are "generated" and can be used
* you can execute the whole notebook by Cell->runAll

Description:
* Used Robot: Planar Manipulator 3DoF
* Benchmark tasks: kin_obst1, kin_obst2
* Planners: BasicPRM, VisibilityPRM, and LazyPRM.

Imports
===========


In [ ]:
import planner_config
import planning
import diagram

In [ ]:
import IPTestSuiteKin3DoF as ts

import matplotlib.pylab as plt
import matplotlib

from IPEnvironmentKin import planarRobotVisualize

Set-up of the test scenario and the configuration for all planner
===================================

Following is a procedure to compare the smoothing algorithms:

1. Configuration for every planner with **benchmark_config.plannerFactory**
2. The configuration and the planner are stored in the variable **plannerFactory**
3. The resulting setup variable is used as a unified interface to execute
   planning calls consistently across all planners.


In [ ]:
plannerFactory = planner_config.plannerFactory

In [ ]:
fullBenchList = ts.benchList
for benchmark in fullBenchList:
    print(benchmark.name)

# Planning and smoothing

Transfer of **plannerFactory** and the list of benchmark tasks to the planning function for pathplanning and subsequent smoothing of the paths

In [ ]:
testList = fullBenchList
resultList = planning.planning(plannerFactory, testList)

# Visualization

Visualization of benchmark tasks

In [ ]:
for benchmark in fullBenchList:
    fig_local = plt.figure(figsize=(7,7))
    ax = fig_local.add_subplot(1,1,1)
    title = benchmark.name
    ax.set_title(title)
    ax.set_xlim([-4,4])
    ax.set_ylim([-4,4])
    try:
        benchmark.collisionChecker.drawObstacles(ax, True)
        benchmark.collisionChecker.kin_chain.move(benchmark.startList[0])
        planarRobotVisualize(benchmark.collisionChecker.kin_chain, ax)
        benchmark.collisionChecker.kin_chain.move(benchmark.goalList[0])
        planarRobotVisualize(benchmark.collisionChecker.kin_chain, ax, 'b')
    except Exception as e:
       print ("Error", e)
       pass

Visualization of the planned paths and the smoothed paths

In [ ]:
matplotlib.rcParams['animation.embed_limit'] = 256
from IPEnvironmentKin import animateSolution
for result in resultList:
    if result.solution != []:
        # Animation Planned Path
        animateSolution(result.planner, result.graph, result.benchmark.collisionChecker, result.solution, plannerFactory[result.plannerFactoryName][2])
        # Animation Smoothing with BG
        animateSolution(result.planner, result.smooth_graph_bg, result.benchmark.collisionChecker, result.smoothed_path_bg, plannerFactory[result.plannerFactoryName][2], title=f"Smooth BG Path\n [{result.plannerFactoryName}]")
        # Animation Random Smoothing
        animateSolution(result.planner, result.smooth_graph_random, result.benchmark.collisionChecker, result.smoothed_path_random, plannerFactory[result.plannerFactoryName][2], title=f"Smooth Random Path\n [{result.plannerFactoryName}]")
    else:
        print(f"{result.plannerFactoryName} {result.benchmark.name} no Path found")

# Evaluation

Diagrams showing the evaluation metrics: path length, number of nodes in the graph, planning and smoothing time

In [ ]:
diagram.generate_diagramm(testList, resultList)

Comparison of the path length and number of nodes for the two smoothing methods

In [ ]:
diagram.generate_timeplot(resultList)

# 5 times planning with LazyPRM
No animation, as the focus is on the diagrams and not on visualizing the paths.

In [ ]:
import IPLazyPRM
import IPVISLazyPRM

In [ ]:
plannerFactory2 = dict()
lazyConfig = dict()
lazyConfig["initialRoadmapSize"] = 40
lazyConfig["updateRoadmapSize"]  = 10
lazyConfig["kNearest"] = 15
lazyConfig["maxIterations"] = 50
plannerFactory2["lazyPRM"] = [IPLazyPRM.LazyPRM, lazyConfig, IPVISLazyPRM.lazyPRMVisualize]
plannerFactory2["lazyPRM2"] = [IPLazyPRM.LazyPRM, lazyConfig, IPVISLazyPRM.lazyPRMVisualize]
plannerFactory2["lazyPRM3"] = [IPLazyPRM.LazyPRM, lazyConfig, IPVISLazyPRM.lazyPRMVisualize]
plannerFactory2["lazyPRM4"] = [IPLazyPRM.LazyPRM, lazyConfig, IPVISLazyPRM.lazyPRMVisualize]
plannerFactory2["lazyPRM5"] = [IPLazyPRM.LazyPRM, lazyConfig, IPVISLazyPRM.lazyPRMVisualize]

Planning and smoothing

In [ ]:
testList = fullBenchList
resultList_lazy = planning.planning(plannerFactory2, testList)

Evaluation

In [ ]:
diagram.generate_diagramm(testList, resultList_lazy)

In [ ]:
diagram.generate_timeplot(resultList_lazy)